# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()

import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
print(con.sql(f"DESCRIBE SELECT * FROM '{fact_path}' LIMIT 1"))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. My rule and its reason codes
**Signal 1 — CTR vs. position tier: CONFIRMED.** Mean CTR declines from top_3 to deep (0.0038 → 0.0004), roughly a 9x drop, with n = 197K–508K in the largest buckets. Median CTR was 0 in every bucket because at this daily grain, 62–93% of rows per tier have zero clicks — so mean_ctr was used instead of median_ctr.

**Signal 2 — CTR vs. impression volume: MIXED.** Mean CTR is nearly flat across buckets (0.0028–0.0031), only a small, noisy decline at higher volume. Not strong enough to anchor a rule on its own.

**Rule (plain words):** Flag content items whose actual CTR is far below the mean CTR of their position tier, among rows with enough impressions (≥50) and usable GSC data (gsc_data_available IS TRUE).

**Reason codes:**
- `ctr_far_below_position_peers` — gap is large (threshold set in Section 2)
- `ctr_below_position_peers` — gap is moderate

**Action label:** `snippet_review`

In [11]:
q_signal = f"""
SELECT
    CASE
        WHEN gsc_sum_position / gsc_impressions <= 3 THEN 'top_3'
        WHEN gsc_sum_position / gsc_impressions <= 10 THEN 'page_1'
        WHEN gsc_sum_position / gsc_impressions <= 20 THEN 'page_2'
        WHEN gsc_sum_position / gsc_impressions <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS position_tier,
    COUNT(*) AS n,
    ROUND(MEDIAN(gsc_clicks * 1.0 / gsc_impressions), 6) AS median_ctr,
    ROUND(AVG(gsc_clicks * 1.0 / gsc_impressions), 6) AS mean_ctr,
    ROUND(SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4) AS zero_click_share
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
  AND gsc_impressions >= 50
GROUP BY position_tier
ORDER BY MIN(gsc_sum_position / gsc_impressions)
"""
signal_result = con.sql(q_signal).df()
print(signal_result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_tier       n  median_ctr  mean_ctr  zero_click_share
0         top_3  197354         0.0  0.003785            0.6213
1        page_1  507890         0.0  0.003347            0.6389
2        page_2  138253         0.0  0.003142            0.7072
3      page_3_5  184964         0.0  0.001566            0.7770
4          deep    8981         0.0  0.000408            0.9316


In [12]:
q_volume = f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN 'low_50_99'
        WHEN gsc_impressions < 500 THEN 'mid_100_499'
        WHEN gsc_impressions < 2000 THEN 'high_500_1999'
        ELSE 'very_high_2000plus'
    END AS volume_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_clicks * 1.0 / gsc_impressions), 6) AS mean_ctr
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
  AND gsc_impressions >= 50
GROUP BY volume_bucket
ORDER BY MIN(gsc_impressions)
"""
volume_result = con.sql(q_volume).df()
print(volume_result)

        volume_bucket       n  mean_ctr
0           low_50_99  398834  0.003070
1         mid_100_499  537157  0.003101
2       high_500_1999   93794  0.002804
3  very_high_2000plus    7657  0.002805


## 2. Build the ranked queue (writes the CSV)

Approach: Aggregated to one row per content item × client (summed across the month), not daily rows, so the queue ranks pages once instead of 30x. Used weighted_position (sum_position/impressions) rather than a plain average, so high-impression days count more. Threshold: only positive ctr_gap rows get a reason code — top decile of positive gaps → ctr_far_below_position_peers, next 40% → ctr_below_position_peers, rest → no flag, action = monitor. This kept the flagged share at ~35% (40,790 of 116,114), instead of ~70% with a looser cutoff — avoiding the "alarm rings all day" problem from the session.

In [13]:
q_queue = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) AS sum_position
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 50
"""
queue_df = con.sql(q_queue).df()

queue_df["ctr"] = queue_df["clicks"] / queue_df["impressions"]
queue_df["weighted_position"] = queue_df["sum_position"] / queue_df["impressions"]

import pandas as pd

queue_df["position_tier"] = pd.cut(
    queue_df["weighted_position"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"]
)

tier_mean_ctr = queue_df.groupby("position_tier", observed=True)["ctr"].mean()
queue_df["expected_ctr"] = queue_df["position_tier"].map(tier_mean_ctr.to_dict()).astype(float)
queue_df["ctr_gap"] = queue_df["expected_ctr"] - queue_df["ctr"]

# Threshold: only positive gaps (underperforming) get a reason code.
# far_below = top quartile of positive gaps; below = rest.
gap_q75 = queue_df.loc[queue_df["ctr_gap"] > 0, "ctr_gap"].quantile(0.75)

def reason_code(gap):
    if gap <= 0:
        return None
    elif gap >= gap_q75:
        return "ctr_far_below_position_peers"
    else:
        return "ctr_below_position_peers"

queue_df["reason_code"] = queue_df["ctr_gap"].apply(reason_code)
queue_df["action"] = queue_df["reason_code"].apply(lambda r: "snippet_review" if r else "monitor")
queue_df["score"] = queue_df["ctr_gap"]

ranked = queue_df.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Total rows in queue: {len(ranked)}")
print(f"Rows flagged (score > 0): {(ranked['score'] > 0).sum()}")
ranked.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows in queue: 116114
Rows flagged (score > 0): 81579


,client_hash_id,content_hash_id,impressions,clicks,sum_position,ctr,weighted_position,position_tier,expected_ctr,ctr_gap,reason_code,action,score
0,client_86ebc2f12c01f586,content_b105ea52750b491c,72.0,0.0,101.0,0.0,1.402778,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
1,client_62f4a7e64f5e0096,content_2eb1b1de0ad1e4c6,339.0,0.0,883.0,0.0,2.604720,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
2,client_62f4a7e64f5e0096,content_19dba65e6e01feb5,299.0,0.0,878.0,0.0,2.936455,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
3,client_73cda7b4e4f265ea,content_365f96e76a5a4a83,233.0,0.0,114.0,0.0,0.489270,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
4,client_62f4a7e64f5e0096,content_c38918b65f634646,518.0,0.0,1257.0,0.0,2.426641,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
5,client_62f4a7e64f5e0096,content_22b90a28f54c7b8c,198.0,0.0,289.0,0.0,1.459596,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
6,client_62f4a7e64f5e0096,content_250bded70cab53e4,188.0,0.0,547.0,0.0,2.909574,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
7,client_62f4a7e64f5e0096,content_a72e1952e9b7d9f5,322.0,0.0,634.0,0.0,1.968944,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
8,client_62f4a7e64f5e0096,content_1727d8aa55dd7d81,486.0,0.0,783.0,0.0,1.611111,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
9,client_62f4a7e64f5e0096,content_817b1e43a1fb4f2c,1772.0,0.0,4132.0,0.0,2.331828,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463


In [14]:
q_queue = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) AS sum_position
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 50
"""
queue_df = con.sql(q_queue).df()

queue_df["ctr"] = queue_df["clicks"] / queue_df["impressions"]
queue_df["weighted_position"] = queue_df["sum_position"] / queue_df["impressions"]

import pandas as pd

queue_df["position_tier"] = pd.cut(
    queue_df["weighted_position"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"]
)

tier_mean_ctr = queue_df.groupby("position_tier", observed=True)["ctr"].mean()
queue_df["expected_ctr"] = queue_df["position_tier"].map(tier_mean_ctr.to_dict()).astype(float)
queue_df["ctr_gap"] = queue_df["expected_ctr"] - queue_df["ctr"]

# Threshold: only positive gaps (underperforming) get a reason code.
# far_below = top decile of positive gaps; below = the rest of the upper half.
gap_q90 = queue_df.loc[queue_df["ctr_gap"] > 0, "ctr_gap"].quantile(0.90)
gap_q50 = queue_df.loc[queue_df["ctr_gap"] > 0, "ctr_gap"].quantile(0.50)

def reason_code(gap):
    if gap <= 0:
        return None
    elif gap >= gap_q90:
        return "ctr_far_below_position_peers"
    elif gap >= gap_q50:
        return "ctr_below_position_peers"
    else:
        return None

queue_df["reason_code"] = queue_df["ctr_gap"].apply(reason_code)
queue_df["action"] = queue_df["reason_code"].apply(lambda r: "snippet_review" if r else "monitor")
queue_df["score"] = queue_df["ctr_gap"]

ranked = queue_df.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Total rows in queue: {len(ranked)}")
print(f"Rows flagged (score > 0, above median gap): {(ranked['action'] == 'snippet_review').sum()}")
ranked.head(20)

Total rows in queue: 116114
Rows flagged (score > 0, above median gap): 40790


,client_hash_id,content_hash_id,impressions,clicks,sum_position,ctr,weighted_position,position_tier,expected_ctr,ctr_gap,reason_code,action,score
0,client_a80fca3f171ed1de,content_46793dce3bd0c958,320.0,0.0,478.0,0.0,1.493750,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
1,client_a80fca3f171ed1de,content_8729604c70b06138,60.0,0.0,166.0,0.0,2.766667,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
2,client_a80fca3f171ed1de,content_61a4d1a153a2633f,137.0,0.0,225.0,0.0,1.642336,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
3,client_62f4a7e64f5e0096,content_42fd52ddb4cda369,106.0,0.0,272.0,0.0,2.566038,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
4,client_62f4a7e64f5e0096,content_ccf40e1244078f49,328.0,0.0,410.0,0.0,1.250000,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
5,client_62f4a7e64f5e0096,content_93715780c06b5596,54.0,0.0,74.0,0.0,1.370370,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
6,client_a80fca3f171ed1de,content_1f17a72b0be029fa,259.0,0.0,774.0,0.0,2.988417,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
7,client_a80fca3f171ed1de,content_f62e7222cad94397,87.0,0.0,239.0,0.0,2.747126,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
8,client_a80fca3f171ed1de,content_b6df90f18c5b4004,93.0,0.0,201.0,0.0,2.161290,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
9,client_a80fca3f171ed1de,content_42dfd9feea49d704,99.0,0.0,255.0,0.0,2.575758,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463


## 3. Top-20 review

All 20 rows sit in position_tier=top_3 with clicks=0, so ctr_gap is identical across all of them (an exact-tie block) — since no tie-break rule has been applied yet, their order within the top 20 is effectively arbitrary. Impressions among these rows vary widely (62 to 10,462): the high-impression, zero-click rows (e.g. content_520e203a08cd69ee at 10,462 impressions) are strong, high-confidence flags — zero clicks at real volume is a genuine snippet problem. The low-impression rows (62-90 impressions) are low-confidence flags — a single click could flip their CTR entirely, so they may just be noise rather than real underperformers. What would make any of these wrong: the page was recently optimized and is still in cooldown, or the query itself is non-clickable (e.g. Google shows a featured snippet directly).

In [15]:
review = ranked.head(20)[[
    "client_hash_id", "content_hash_id", "impressions", "clicks",
    "ctr", "weighted_position", "position_tier",
    "expected_ctr", "ctr_gap", "reason_code", "action"
]].copy()

print(review.to_string(index=True))


             client_hash_id           content_hash_id  impressions  clicks  ctr  weighted_position position_tier  expected_ctr   ctr_gap                   reason_code          action
0   client_a80fca3f171ed1de  content_46793dce3bd0c958        320.0     0.0  0.0           1.493750         top_3      0.003463  0.003463  ctr_far_below_position_peers  snippet_review
1   client_a80fca3f171ed1de  content_8729604c70b06138         60.0     0.0  0.0           2.766667         top_3      0.003463  0.003463  ctr_far_below_position_peers  snippet_review
2   client_a80fca3f171ed1de  content_61a4d1a153a2633f        137.0     0.0  0.0           1.642336         top_3      0.003463  0.003463  ctr_far_below_position_peers  snippet_review
3   client_62f4a7e64f5e0096  content_42fd52ddb4cda369        106.0     0.0  0.0           2.566038         top_3      0.003463  0.003463  ctr_far_below_position_peers  snippet_review
4   client_62f4a7e64f5e0096  content_ccf40e1244078f49        328.0     0.0  0.0      

## 4. Weak picks + leakage check

**Weak picks:** 7,237 of the 40,790 flagged rows (~18%) have impressions under 100 — all sitting in the same zero-click, top_3 tie block (ctr_gap = 0.003463 for every one). At this volume a single click would flip their CTR entirely, so these are low-confidence flags rather than confirmed snippet problems. In a real workflow these should either be excluded by raising the minimum-impression threshold, or reviewed last, after the high-volume flags.

**Leakage check:** All columns feeding the score (impressions, clicks, sum_position, ctr, weighted_position, position_tier, expected_ctr, ctr_gap) are aggregated from the same month (2026-03) only — no future window was used. client_hash_id and content_hash_id are grouping keys only, never fed into the score. ctr_gap is derived from same-period ctr and expected_ctr, not from any product flag or external label.

In [16]:
# Weak picks: flagged rows with very low impressions (single click could flip their CTR)
weak_picks = ranked[
    (ranked["action"] == "snippet_review") &
    (ranked["impressions"] < 100)
].sort_values("score", ascending=False)

print(f"Flagged rows with impressions < 100: {len(weak_picks)}")
print(weak_picks.head(10)[["client_hash_id", "content_hash_id", "impressions", "clicks", "ctr_gap"]])

# Leakage check: list every column that fed the score, confirm none are future-window or label-derived
print("\nColumns used to build the rule/score:")
print(list(queue_df.columns))


Flagged rows with impressions < 100: 7237
               client_hash_id           content_hash_id  impressions  clicks  \
2031  client_fef1a8f436438636  content_21cf8ef6595b585d         60.0     0.0   
2037  client_73cda7b4e4f265ea  content_1fc85ecb99f47d57         80.0     0.0   
2043  client_73cda7b4e4f265ea  content_25160951304ccb6b         80.0     0.0   
2047  client_fef1a8f436438636  content_8dbdb6667a2fd783         76.0     0.0   
2048  client_fef1a8f436438636  content_b92c155429cfcd98         79.0     0.0   
2050  client_73cda7b4e4f265ea  content_f9404a36708033d1         52.0     0.0   
2057  client_fef1a8f436438636  content_4cce62cba7ac9ea3         54.0     0.0   
2061  client_fef1a8f436438636  content_6b00cdce458cef11         84.0     0.0   
1362  client_fef1a8f436438636  content_c6b91202a9d18ec4         90.0     0.0   
2076  client_fef1a8f436438636  content_4df357141fc667b5         87.0     0.0   

       ctr_gap  
2031  0.003463  
2037  0.003463  
2043  0.003463  
2047  0.0

## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.